In [25]:
import numpy as np 
import tensorflow as tf 
import wave, math, struct
from keras.models import Sequential
from keras.layers import Dense, LSTM, GRU, Activation, Input

In [5]:
# Prepare a dummy data for simulating musical notes 
notes_freqs = {
    'A': 440.0, 'B':493.88, 'C':261.63, 'D':293.66, 'E':393.63, 
    'F': 349.23, 'G': 392.0
}

In [7]:
notes_freqs;

In [8]:
notes = list(notes_freqs.keys())
notes

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [9]:
note_to_int = {note: i for i, note in enumerate(notes)}

In [10]:
note_to_int

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}

In [11]:
int_to_note = {i: note for i, note in enumerate(notes)}
int_to_note

{0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

In [12]:
raw_music_data = [notes[np.random.randint(0,7)] for i in range(1000)]

In [13]:
raw_music_data

['D',
 'E',
 'D',
 'A',
 'C',
 'B',
 'F',
 'F',
 'D',
 'B',
 'B',
 'D',
 'B',
 'B',
 'G',
 'G',
 'C',
 'B',
 'G',
 'E',
 'G',
 'E',
 'G',
 'F',
 'F',
 'E',
 'E',
 'F',
 'B',
 'F',
 'F',
 'B',
 'B',
 'D',
 'C',
 'B',
 'D',
 'C',
 'C',
 'C',
 'F',
 'D',
 'F',
 'A',
 'G',
 'B',
 'G',
 'F',
 'D',
 'B',
 'D',
 'E',
 'F',
 'B',
 'D',
 'G',
 'C',
 'D',
 'B',
 'G',
 'D',
 'C',
 'F',
 'A',
 'F',
 'D',
 'E',
 'D',
 'C',
 'D',
 'D',
 'E',
 'C',
 'G',
 'E',
 'G',
 'G',
 'D',
 'C',
 'C',
 'F',
 'D',
 'A',
 'B',
 'F',
 'G',
 'C',
 'D',
 'D',
 'D',
 'A',
 'E',
 'B',
 'C',
 'B',
 'G',
 'G',
 'B',
 'F',
 'B',
 'A',
 'D',
 'E',
 'D',
 'E',
 'B',
 'G',
 'A',
 'A',
 'B',
 'F',
 'G',
 'E',
 'C',
 'G',
 'D',
 'E',
 'D',
 'C',
 'E',
 'B',
 'C',
 'A',
 'C',
 'G',
 'B',
 'E',
 'E',
 'F',
 'A',
 'A',
 'E',
 'C',
 'A',
 'B',
 'G',
 'D',
 'B',
 'A',
 'E',
 'A',
 'B',
 'D',
 'C',
 'F',
 'C',
 'B',
 'A',
 'D',
 'G',
 'F',
 'D',
 'F',
 'G',
 'G',
 'G',
 'G',
 'C',
 'B',
 'E',
 'F',
 'G',
 'F',
 'G',
 'G',
 'F',
 'A'

#### Data Preparation

In [15]:
seq_length = 3
network_input = []
network_output = []

for i in range(len(raw_music_data) - seq_length): 
    seq_in = raw_music_data[i: i+seq_length]
    seq_out = raw_music_data[i+seq_length]
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])
    print(seq_in,'-->',seq_out)

['D', 'E', 'D'] --> A
['E', 'D', 'A'] --> C
['D', 'A', 'C'] --> B
['A', 'C', 'B'] --> F
['C', 'B', 'F'] --> F
['B', 'F', 'F'] --> D
['F', 'F', 'D'] --> B
['F', 'D', 'B'] --> B
['D', 'B', 'B'] --> D
['B', 'B', 'D'] --> B
['B', 'D', 'B'] --> B
['D', 'B', 'B'] --> G
['B', 'B', 'G'] --> G
['B', 'G', 'G'] --> C
['G', 'G', 'C'] --> B
['G', 'C', 'B'] --> G
['C', 'B', 'G'] --> E
['B', 'G', 'E'] --> G
['G', 'E', 'G'] --> E
['E', 'G', 'E'] --> G
['G', 'E', 'G'] --> F
['E', 'G', 'F'] --> F
['G', 'F', 'F'] --> E
['F', 'F', 'E'] --> E
['F', 'E', 'E'] --> F
['E', 'E', 'F'] --> B
['E', 'F', 'B'] --> F
['F', 'B', 'F'] --> F
['B', 'F', 'F'] --> B
['F', 'F', 'B'] --> B
['F', 'B', 'B'] --> D
['B', 'B', 'D'] --> C
['B', 'D', 'C'] --> B
['D', 'C', 'B'] --> D
['C', 'B', 'D'] --> C
['B', 'D', 'C'] --> C
['D', 'C', 'C'] --> C
['C', 'C', 'C'] --> F
['C', 'C', 'F'] --> D
['C', 'F', 'D'] --> F
['F', 'D', 'F'] --> A
['D', 'F', 'A'] --> G
['F', 'A', 'G'] --> B
['A', 'G', 'B'] --> G
['G', 'B', 'G'] --> F
['B', 'G',

In [16]:
n_paterns = len(network_input)

In [17]:
n_paterns

997

In [18]:
x = np.reshape(network_input,(n_paterns, seq_length, 1))
x

array([[[3],
        [4],
        [3]],

       [[4],
        [3],
        [0]],

       [[3],
        [0],
        [2]],

       ...,

       [[2],
        [0],
        [0]],

       [[0],
        [0],
        [0]],

       [[0],
        [0],
        [4]]])

In [19]:
from keras.utils import to_categorical

In [20]:
y = to_categorical(network_output)

In [21]:
y.shape

(997, 7)

In [22]:
y

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]])

#### Build the model

In [26]:
model = Sequential()
model.add(Input((3,1)))
model.add(GRU(256))
model.add(Dense(512, activation='relu'))
model.add(Dense(7,activation='softmax'))

In [27]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 256)            │       198,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 334,087 (1.27 MB)

 Trainable params: 334,087 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
model.compile(loss='categorical_crossentropy', 
              optimizer='adam', metrics=['accuracy'])

In [31]:
model.fit(x, y, epochs=1000, batch_size=10)

Epoch 1/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3310 - loss: 1.6281
Epoch 2/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3213 - loss: 1.6329
Epoch 3/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3260 - loss: 1.6295
Epoch 4/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3395 - loss: 1.6503
Epoch 5/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3609 - loss: 1.6552
Epoch 6/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3519 - loss: 1.6397
Epoch 7/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3353 - loss: 1.6235
Epoch 8/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3505 - loss: 1.5920
Epoch 9/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3613 - loss: 1.5876
Epoch 10/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3540 - loss: 1.6142
Epoch 11/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3225 - loss: 1.6359
Epoch 12/1000
100/100 ━━━━━━━━

#### Generate new melody sequence

In [43]:
start_index = np.random.randint(0, len(network_output))
pattern = network_input[start_index]
pattern

[4, 1, 2]

In [55]:
generated_melody = [] 
for i in range(16): 
    x_input = np.reshape(pattern, (1,len(pattern), 1))
    pred = model.predict(x_input, verbose=False)
    index = np.argmax(pred) 
    result = int_to_note[index]
    generated_melody.append(result)
    pattern.append(index)
    pattern = pattern[1:len(pattern)]

In [56]:
generated_melody

['F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F',
 'F']

#### Save this as audio file

In [50]:
with wave.open('my_music.wav','w') as wav_file: 
    wav_file.setparams((1,2,44100,0,'NONE','not compressed'))
    for note in generated_melody: 
        freq = notes_freqs[note]
        num_samples = int(0.5 * 44100) 
        for i in range(num_samples): 
            t = float(i) / 44100
            value = int(32767 * 0.5 * math.sin(2*math.pi*freq*t))
            data = struct.pack('<h', value)
            wav_file.writeframes(data)